# Seed-Level Case Studies For Drift Detection

This notebook generates representative single-seed timelines to complement multi-seed aggregate results.

Selection rule (predefined, non-cherry-picked):
- earliest warning lead
- median warning lead
- latest warning lead

for each selected experiment.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Config ---
DRIFT_START = 5              # synthetic experiments: first drifted window (0-based)
SMOOTH_WINDOW = 3            # matches lead-time convention used in main analysis
THRESHOLD_STD = 2.0

EXPERIMENTS = [
    "Covariate",
    "Concept (class-cond)",
    "Mixed",
]

MANUAL_POSITIVE_SELECTION = {
    "Covariate": {"positive_a": 4, "positive_b": 15},
    "Concept (class-cond)": {"positive_a": 0, "positive_b": 13},
    "Mixed": {"positive_a": 0, "positive_b": 13},
}

RESULTS_ROOT = Path("../results")
RUN_ID = (RESULTS_ROOT / "latest_run.txt").read_text().strip()
RUN_ROOT = RESULTS_ROOT / "runs" / RUN_ID
TABLE_PATH = RUN_ROOT / "tables" / "full_results.csv"
FIG_DIR = RUN_ROOT / "figures" / "seed_case_studies"
FIG_DIR.mkdir(parents=True, exist_ok=True)

full_df = pd.read_csv(TABLE_PATH)
print(f"Run: {RUN_ID}")
print(f"Loaded rows: {len(full_df)} from {TABLE_PATH}")


In [ ]:
def first_warning_window(alert_levels: list[str], windows: np.ndarray) -> int | None:
    for w, level in zip(windows, alert_levels):
        if level in {"warning", "critical"}:
            return int(w)
    return None


def first_accuracy_drop_window(
    accuracy: np.ndarray,
    windows: np.ndarray,
    drift_start: int = DRIFT_START,
    smooth_window: int = SMOOTH_WINDOW,
    threshold_std: float = THRESHOLD_STD,
) -> int | None:
    if len(accuracy) == 0:
        return None

    pre_mask = windows < drift_start
    pre = accuracy[pre_mask]
    if len(pre) == 0:
        pre = accuracy[: max(3, len(accuracy) // 4)]

    thresh = pre.mean() - threshold_std * (pre.std() + 1e-10)
    smooth = pd.Series(accuracy).rolling(smooth_window, min_periods=1, center=True).mean().to_numpy()
    idx = np.where(smooth < thresh)[0]
    if len(idx) == 0:
        return None
    return int(windows[idx[0]])


def warning_lead_time_for_seed(seed_df: pd.DataFrame) -> float | None:
    g = seed_df.sort_values("window")
    windows = g["window"].to_numpy(dtype=int)
    alerts = g["alert_level"].astype(str).str.lower().tolist()
    acc = g["accuracy"].to_numpy(dtype=float)

    warn_w = first_warning_window(alerts, windows)
    acc_w = first_accuracy_drop_window(acc, windows)

    if warn_w is None or acc_w is None:
        return None
    return float(acc_w - warn_w)


In [ ]:
def select_representative_seeds(exp_df: pd.DataFrame) -> dict[str, int]:
    rows = []
    for seed, g in exp_df.groupby("seed"):
        lead = warning_lead_time_for_seed(g)
        if lead is not None:
            rows.append((int(seed), float(lead)))

    if not rows:
        return {}

    leads = pd.DataFrame(rows, columns=["seed", "lead"]).sort_values("lead").reset_index(drop=True)

    earliest_seed = int(leads.iloc[0]["seed"])
    latest_seed = int(leads.iloc[-1]["seed"])
    median_target = float(leads["lead"].median())
    median_idx = (leads["lead"] - median_target).abs().idxmin()
    median_seed = int(leads.loc[median_idx, "seed"])

    return {
        "earliest": earliest_seed,
        "median": median_seed,
        "latest": latest_seed,
    }


def format_exp_name(name: str) -> str:
    return (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )


In [ ]:
def plot_seed_timeline(exp_name: str, seed: int, role: str, exp_df: pd.DataFrame) -> Path:
    g = exp_df[exp_df["seed"] == seed].sort_values("window")
    windows = g["window"].to_numpy(dtype=int)

    acc = g["accuracy"].to_numpy(dtype=float)
    cosine = g["cosine_drift"].to_numpy(dtype=float)
    max_jsd = g["max_jsd"].to_numpy(dtype=float)
    max_w = g["max_wasserstein"].to_numpy(dtype=float)
    alerts = g["alert_level"].astype(str).str.lower().tolist()

    warn_w = first_warning_window(alerts, windows)
    acc_w = first_accuracy_drop_window(acc, windows)

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # background status bands by alert level
    alert_colors = {
        "ok": "#2ecc71",
        "warning": "#f39c12",
        "critical": "#e74c3c",
    }
    for w, level in zip(windows, alerts):
        color = alert_colors.get(level, "#bdc3c7")
        ax1.axvspan(w - 0.5, w + 0.5, color=color, alpha=0.18, zorder=0)

    ax1.plot(windows, cosine, marker="o", label="cosine_drift", color="#1f77b4")
    ax1.plot(windows, max_jsd, marker="o", label="max_jsd", color="#2ca02c")
    ax1.plot(windows, max_w, marker="o", label="max_wasserstein", color="#ff7f0e")
    ax1.set_xlabel("Window")
    ax1.set_ylabel("Drift Metric")

    ax2 = ax1.twinx()
    ax2.plot(windows, acc, color="#d62728", linestyle="--", linewidth=2, label="accuracy")
    ax2.set_ylabel("Accuracy", color="#d62728")
    ax2.tick_params(axis="y", labelcolor="#d62728")

    if DRIFT_START is not None:
        ax1.axvline(DRIFT_START, color="gray", linestyle=":", alpha=0.8, label="drift start")
    if warn_w is not None:
        ax1.axvline(warn_w, color="#f39c12", linestyle="--", alpha=0.9, label="first warning")
    if acc_w is not None:
        ax1.axvline(acc_w, color="#2980b9", linestyle="-.", alpha=0.9, label="accuracy drop")

    lead_str = "N/A" if (warn_w is None or acc_w is None) else str(acc_w - warn_w)
    ax1.set_title(f"{exp_name} | seed={seed} ({role}) | warning lead={lead_str}")

    # legend: metric lines + status patches + accuracy line
    from matplotlib.patches import Patch
    status_patches = [
        Patch(facecolor="#2ecc71", alpha=0.18, label="OK window"),
        Patch(facecolor="#f39c12", alpha=0.18, label="WARNING window"),
        Patch(facecolor="#e74c3c", alpha=0.18, label="CRITICAL window"),
    ]

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(status_patches + h1 + h2, [p.get_label() for p in status_patches] + l1 + l2, loc="upper right", fontsize=8)

    fig.tight_layout()

    out = FIG_DIR / f"{format_exp_name(exp_name)}_seed{seed}_{role}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out


In [ ]:
summary_rows = []
output_paths = []

for exp_name in EXPERIMENTS:
    exp_df = full_df[full_df["experiment"] == exp_name].copy()
    if exp_df.empty:
        print(f"Skipping {exp_name}: no rows")
        continue

    picks = MANUAL_POSITIVE_SELECTION.get(exp_name, {})
    if not picks:
        print(f"Skipping {exp_name}: no manual seeds configured")
        continue

    for role, seed in picks.items():
        path = plot_seed_timeline(exp_name, seed, role, exp_df)
        output_paths.append(path)
        lead = warning_lead_time_for_seed(exp_df[exp_df["seed"] == seed])
        summary_rows.append({
            "experiment": exp_name,
            "role": role,
            "seed": seed,
            "warning_lead": lead,
            "figure": str(path),
        })

summary_df = pd.DataFrame(summary_rows).sort_values(["experiment", "role"])
summary_csv = FIG_DIR / "case_study_seed_selection.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Saved {len(output_paths)} figures to: {FIG_DIR}")
print(f"Selection summary: {summary_csv}")
summary_df
